In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
from pathlib import Path

# time series decomposition
from statsmodels.tsa.seasonal import seasonal_decompose

# anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from scipy import stats

# llm for the report agent
import anthropic

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [6]:
DATA_DIR = Path('project_anomaly_detection')

# load both datasets
viaggiatori = pd.read_csv(DATA_DIR / 'TIPOLOGIA_VIAGGIATORE.csv')
allarmi = pd.read_csv(DATA_DIR / 'ALLARMI.csv')

print('viaggiatori:', viaggiatori.shape)
print('allarmi:', allarmi.shape)

viaggiatori: (5095, 33)
allarmi: (5080, 24)


In [7]:
# quick look at the traveler dataset
viaggiatori.head()

,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,CODICE_PAESE_ARR,CODICE_PAESE_PART,PAESE_ARR,PAESE_PART,ZONA,ENTRATI,INVESTIGATI,ALLARMATI,TIPO_DOCUMENTO,GENERE,FASCIA_ETA,FLAG_TRANSITO,COMPAGNIA_AEREA,NUMERO_VOLO,ESITO_CONTROLLO,note_operatore,codice_rischio,Tipo Documento,FASCIA ETA,3nazionalita,compagnia%aerea,num volo
0,ALB,NAP,DUR,2024,02,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,Durban,ITA,ZAF,Italia,Sudafrica,6,1,1,0,Passaporto,F,N.D.,Singola Tratta,Fly Dubai,FZ1681,RESPINTO,NaN,NaN,Passaporto,N.D.,ALB,Fly Dubai,FZ1681
1,NaN,FCO,JFK,2024,01,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,New York,ITA,USA,Italia,Stati Uniti,5,1,0,1,Carta d'identità,F,18-30,Singola Tratta,ITA Airways,AZ0609,NaN,NaN,NaN,Carta d'identità,18-30,ALB,ITA Airways,AZ0609
2,ALB,TSF,TIA,2024,02,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,Tirana,ITA,ALB,Italia,Albania,4,58,58,13,N.D.,f,31-45,Singola Tratta,Ryanair DAC,FR8400,SEGNALATO,NaN,NaN,N.D.,31-45,ALB,Ryanair DAC,FR8400
3,AFG,FCO,IST,2024,01,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,Istanbul,IT,TR,Italia,Turchia,5,1,1,0,N.D.,M,61+,Singola Tratta,Turkish Airlines,TK1865,NaN,NaN,NaN,N.D.,61+,AFG,Turkish Airlines,TK1865
4,ALB,BGY,MLE,2024,02,13,FEB 13 2024,Orio al Serio,Male International,Bergamo,Male,ITA,MDV,Italia,Maldive,2,2,2,1,Permesso di soggiorno,F,46-60,Singola Tratta,Fly Dubai,FZ1571,SEGNALATO,NaN,NaN,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571


In [8]:
viaggiatori.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5095 entries, 0 to 5094
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   NAZIONALITA            4979 non-null   object
 1   AREOPORTO_ARRIVO       5095 non-null   object
 2   AREOPORTO_PARTENZA     5095 non-null   object
 3   ANNO_PARTENZA          5095 non-null   object
 4   MESE_PARTENZA          5095 non-null   object
 5   GIORNO_PARTENZA        5095 non-null   int64 
 6   DATA_PARTENZA          5095 non-null   object
 7   DESCR_AEREOPORTO_ARR   5095 non-null   object
 8   DESCR_AEREOPORTO_PART  5095 non-null   object
 9   CITTA_ARR              5095 non-null   object
 10  CITTA_PARTENZA         5095 non-null   object
 11  CODICE_PAESE_ARR       5095 non-null   object
 12  CODICE_PAESE_PART      5095 non-null   object
 13  PAESE_ARR              5095 non-null   object
 14  PAESE_PART             5095 non-null   object
 15  ZONA                 

In [9]:
# quick look at the alarms dataset
allarmi.head()

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,CODICE_PAESE_ARR,CODICE_PAESE_PART,PAESE_ARR,PAESE_PART,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,ITA,TUR,Italia,Turchia,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,ITA,GBR,Italia,Regno Unito,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,ITA,GBR,Italia,Regno Unito,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,IT,GB,Italia,Regno Unito,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,ITA,GBR,Italia,Regno Unito,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2


In [10]:
allarmi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5080 entries, 0 to 5079
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   OCCORRENZE             5080 non-null   object
 1   AREOPORTO_ARRIVO       5080 non-null   object
 2   AREOPORTO_PARTENZA     5080 non-null   object
 3   ANNO_PARTENZA          5080 non-null   object
 4   MESE_PARTENZA          5080 non-null   object
 5   DATA_PARTENZA          5080 non-null   object
 6   DESCR_AEREOPORTO_ARR   5080 non-null   object
 7   DESCR_AEREOPORTO_PART  4971 non-null   object
 8   CITTA_ARR              5080 non-null   object
 9   CITTA_PARTENZA         4979 non-null   object
 10  CODICE_PAESE_ARR       5080 non-null   object
 11  CODICE_PAESE_PART      5026 non-null   object
 12  PAESE_ARR              5080 non-null   object
 13  PAESE_PART             5006 non-null   object
 14  ZONA                   5080 non-null   object
 15  TOT                  

## Cleaning - TIPOLOGIA_VIAGGIATORE

In [12]:
df = viaggiatori.copy()

# all the ways null is encoded in this dataset, found in the dataset all the possible variant to NULL
NULL_VALUES = ['N.D.', 'n.d.', 'ND', 'N/D', 'N/A', 'n/a', '?', '??', '???',
               '//', '-', 'null', 'NULL', 'unknown', 'Unknown', 'UNKN', 'UNK',
               ' ', '', 'ZZ', 'XX', 'EU']

df.replace(NULL_VALUES, np.nan, inplace=True)

# the last 5 columns are helper/cleaned versions added externally, drop them
df.drop(columns=['Tipo Documento', 'FASCIA ETA', '3nazionalita', 'compagnia%aerea', 'num volo'], inplace=True)

print(df.shape)

(5095, 28)


In [17]:
# fix ANNO: '24' and 'anno 2024' and '2023' (data entry errors) all become 2024
df['ANNO_PARTENZA'] = 2024

# fix MESE: Italian text months to numeric
italian_months = {'GEN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAG': 5, 'GIU': 6,
                  'LUG': 7, 'AGO': 8, 'SET': 9, 'OTT': 10, 'NOV': 11, 'DIC': 12}
df['MESE_PARTENZA'] = df['MESE_PARTENZA'].replace(italian_months).astype(float).astype('Int64')

# parse DATA_PARTENZA: dataset has many different formats mixed together
italian_months_long = {'GEN': 'Jan', 'FEB': 'Feb', 'MAR': 'Mar', 'APR': 'Apr',
                       'MAG': 'May', 'GIU': 'Jun', 'LUG': 'Jul', 'AGO': 'Aug',
                       'SET': 'Sep', 'OTT': 'Oct', 'NOV': 'Nov', 'DIC': 'Dec'}

# function to parse dates, used to replace all the strange value in the various column dates. E.G. FEB 13 2024 now becomes 2024-02-13
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    for ita, eng in italian_months_long.items():
        val = val.replace(ita, eng)
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%Y-%m-%d',
                '%Y/%m/%d', '%d.%m.%Y', '%d-%m-%y', '%b %d %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    try:
        return pd.to_datetime(val, dayfirst=True)
    except:
        return pd.NaT

df['DATA_PARTENZA'] = df['DATA_PARTENZA'].apply(parse_date)

print(df['DATA_PARTENZA'].isna().sum(), 'unparsed dates')
print(df['DATA_PARTENZA'].dtype)
print(df['DATA_PARTENZA'].head(3))

0 unparsed dates
datetime64[ns]
0   2024-02-13 07:30:00
1   2024-01-22 16:35:00
2   2024-02-04 20:10:00
Name: DATA_PARTENZA, dtype: datetime64[ns]


### Cleaning numeric columns: ENTRATI, INVESTIGATI, ALLARMATI

These three columns should be integers but contain text artifacts from manual data entry. Examples of dirty values found:

| Raw value | Issue | Cleaned |
|-----------|-------|---------|
| `"17 pax"` | unit label attached | `17.0` |
| `"~0"` | approximate symbol | `0.0` |
| `"102,0"` | comma as decimal separator | `102.0` |
| `"N.D."` | already NaN from previous step | `NaN` |

The `extract_number` function strips these artifacts and converts to float. Any remaining negative values are treated as input errors and set to NaN.

In [21]:
# clean ENTRATI, INVESTIGATI, ALLARMATI: strip text artifacts, keep the number
def extract_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = val.replace(',', '.').replace('~', '').replace('pax', '').strip()
    try:
        return float(val)
    except:
        return np.nan

df['ENTRATI']    = df['ENTRATI'].apply(extract_number)
df['INVESTIGATI'] = df['INVESTIGATI'].apply(extract_number)
df['ALLARMATI']  = df['ALLARMATI'].apply(extract_number)

# negative or zero investigators is impossible, treat as null
df.loc[df['INVESTIGATI'] < 0, 'INVESTIGATI'] = np.nan
df.loc[df['ALLARMATI'] < 0, 'ALLARMATI'] = np.nan

print(df[['ENTRATI', 'INVESTIGATI', 'ALLARMATI']].describe())

            ENTRATI  INVESTIGATI    ALLARMATI
count   5010.000000  5008.000000  5017.000000
mean      42.266267    41.564497     7.466813
std      248.559256   227.369899    71.786689
min     -500.000000     0.000000     0.000000
25%        1.000000     1.000000     0.000000
50%        3.000000     2.000000     1.000000
75%       76.000000    75.000000    10.000000
max    10000.000000  9999.000000  5000.000000


### Standardizing categorical columns

Four cleaning operations in this cell:

**1. GENERE → M / F / NaN**  
The column has many variants for the same value (`f`, `Femmina`, `Female` → `F`; `m`, `Maschio`, `Male` → `M`). Anything that doesn't map to either is set to NaN (e.g. `X`, `N/B`, `1`, `2`).

**2. Airport codes → uppercase**  
IATA codes like `Bgy`, `vce`, `Pmo` are standardized to `BGY`, `VCE`, `PMO`. This is needed later when grouping by route.

**3. Country codes: 2-letter → 3-letter (ISO 3166-1 alpha-3)**  
Some rows use the 2-letter standard (`IT`, `AL`, `GB`) and others use 3-letter (`ITA`, `ALB`, `GBR`). We unify everything to alpha-3 to avoid treating the same country as two different values.

| Alpha-2 | Alpha-3 |
|---------|---------|
| IT | ITA |
| AL | ALB |
| TR | TUR |
| AE | ARE |
| GB | GBR |

**4. NAZIONALITA → valid 3-letter code or NaN**  
After uppercasing, any value that is not exactly 3 characters long is invalid (e.g. `//`, `EU`, `UNKN`) and is set to NaN.

In [ ]:
# normalize GENERE to M/F/NaN
female_vals = {'f', 'femmina', 'female', 'donna', 'f.'}
male_vals   = {'m', 'maschio', 'male', 'uomo', 'male'}

def normalize_gender(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()
    if v in female_vals:
        return 'F'
    if v in male_vals:
        return 'M'
    return np.nan

df['GENERE'] = df['GENERE'].apply(normalize_gender)

# uppercase airport codes and fix mixed case
df['AREOPORTO_ARRIVO'] = df['AREOPORTO_ARRIVO'].str.strip().str.upper()
df['AREOPORTO_PARTENZA'] = df['AREOPORTO_PARTENZA'].str.strip().str.upper()

# fix 2-letter ISO country codes to 3-letter
iso2_to_iso3 = {'IT': 'ITA', 'AL': 'ALB', 'TR': 'TUR', 'AE': 'ARE',
                'GB': 'GBR', 'EG': 'EGY', 'DE': 'DEU', 'FR': 'FRA'}

df['CODICE_PAESE_ARR']  = df['CODICE_PAESE_ARR'].str.strip().str.upper().replace(iso2_to_iso3)
df['CODICE_PAESE_PART'] = df['CODICE_PAESE_PART'].str.strip().str.upper().replace(iso2_to_iso3)

# clean NAZIONALITA: strip and uppercase, anything not a valid 3-letter code becomes NaN
df['NAZIONALITA'] = df['NAZIONALITA'].str.strip().str.upper()
df.loc[df['NAZIONALITA'].str.len() != 3, 'NAZIONALITA'] = np.nan

print('GENERE:', df['GENERE'].value_counts(dropna=False).to_dict())
print('CODICE_PAESE_ARR unique:', df['CODICE_PAESE_ARR'].unique())
print('NAZIONALITA nulls:', df['NAZIONALITA'].isna().sum())

GENERE: {'F': 2226, 'M': 2195, nan: 674}
CODICE_PAESE_ARR unique: ['ITA']
NAZIONALITA nulls: 321


In [ ]:
# final check: null counts per column
null_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].round(1).to_string())
print('cleaned shape:', df.shape)

viaggiatori_clean = df.copy()